# Crowd-safety dataset preparation

This notebook keeps the four approved video sources under Google Drive and writes deterministic binary manifests. It never copies videos into split folders; split membership lives only in `train.json`, `val.json`, `test.json`, and `external_test.json`.

Run the media-free contract check first. The canonical Drive dataset root is `MyDrive/crowd_safety/datasets/`; UBI-Fights downloads from Kaggle, Surveillance Fight clones the supplied GitHub repository, and Violent-Flows is read from `MyDrive/crowd_safety/raw/violent_flows/` by default. Set source environment variables only to override these sources.

For SCVD and UBI-Fights, the setup cell downloads the configured Kaggle datasets when a Colab Secret named `KAGGLE_API_TOKEN` is available. It loads the token without printing it, installs the Kaggle CLI when needed, and limits canonical Drive video storage to 15 GB or the current free space, whichever is lower. Set `RUN_WORKFLOW = True` only in an authorised Colab session.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json
import os
import sys
import random
import shutil
import subprocess
import tempfile

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass
try:
    from google.colab import userdata
    kaggle_token = userdata.get('KAGGLE_API_TOKEN')
    if kaggle_token:
        os.environ['KAGGLE_API_TOKEN'] = kaggle_token
except Exception:
    pass
if shutil.which('kaggle') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/crowd_safety')
DATASETS_ROOT = DRIVE_ROOT / 'datasets'
MANIFESTS_ROOT = DRIVE_ROOT / 'manifests'
SPLIT_SEED = 42
RUN_WORKFLOW = False
MAX_DATASET_GB = 15.0
MAX_DATASET_BYTES = int(MAX_DATASET_GB * 10**9)
DRIVE_FREE_RESERVE_GB = 0.25
DRIVE_FREE_RESERVE_BYTES = int(DRIVE_FREE_RESERVE_GB * 10**9)
VIDEO_SUFFIXES = {'.avi', '.mkv', '.mov', '.mp4', '.mpeg', '.mpg', '.webm'}
DATASET_REGISTRY = {
    'violent_flows': {'role': 'external_test', 'classes': {'violence': 'violent', 'nonviolence': 'normal', 'nonviolent': 'normal', 'non_violence': 'normal'}},
    'ubi_fights': {'role': 'train_val_test', 'classes': {'violence': 'violent', 'violent': 'violent', 'fight': 'violent', 'nonviolence': 'normal', 'nonviolent': 'normal', 'non_violent': 'normal', 'normal': 'normal', 'nofight': 'normal'}},
    'surveillance_fight': {'role': 'train_val_test', 'classes': {'violence': 'violent', 'violent': 'violent', 'fight': 'violent', 'nonviolence': 'normal', 'nonviolent': 'normal', 'non_violent': 'normal', 'normal': 'normal', 'nofight': 'normal'}},
    'scvd': {'role': 'train_val_test', 'classes': {'normal': 'normal', 'violence': 'violent'}},
}
VIOLENT_FLOWS_DEFAULT_SOURCE = DRIVE_ROOT / 'raw/violent_flows'
SCVD_KAGGLE_DATASET = os.environ.get('SCVD_KAGGLE_DATASET', 'toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd')
UBI_FIGHTS_KAGGLE_DATASET = os.environ.get('UBI_FIGHTS_KAGGLE_DATASET', 'intissarziani/ubi-fightsall')
SURVEILLANCE_FIGHT_GITHUB_REPO = 'https://github.com/seymanurakti/fight-detection-surv-dataset.git'
print({'drive_root': str(DRIVE_ROOT), 'manifest_root': str(MANIFESTS_ROOT), 'seed': SPLIT_SEED, 'dataset_budget_gb': MAX_DATASET_GB, 'datasets': tuple(DATASET_REGISTRY)})

## Source import and discovery

Violent-Flows is evaluation-only. UBI-Fights and SCVD download from their configured Kaggle dataset IDs when credentials are available; Surveillance Fight clones the configured GitHub repository unless an override source directory is provided. New video copies are selected deterministically until the 15 GB dataset budget (or current Drive free space, if lower) is reached.

In [ ]:
def _norm(value):
    return ''.join(ch for ch in str(value).casefold() if ch.isalnum())

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _video_files(root):
    return sorted(path for path in Path(root).rglob('*') if path.is_file() and path.suffix.casefold() in VIDEO_SUFFIXES)

def _dataset_bytes():
    return sum(path.stat().st_size for path in _video_files(DATASETS_ROOT))

def _effective_dataset_budget():
    existing = _dataset_bytes()
    drive_free = shutil.disk_usage(DRIVE_ROOT).free if DRIVE_ROOT.exists() else MAX_DATASET_BYTES
    return min(MAX_DATASET_BYTES, existing + max(0, drive_free - DRIVE_FREE_RESERVE_BYTES))

def _class_for(path, dataset):
    aliases = DATASET_REGISTRY[dataset]['classes']
    matches = []
    for part in (path, *path.parents):
        label = aliases.get(_norm(part.name))
        if label is not None:
            matches.append((part.name, label))
    if not matches:
        return None, None
    if len({label for _, label in matches}) != 1:
        raise ValueError(f'ambiguous class layout for {path}')
    return matches[0][1], matches[0][0]

def _official_split(path):
    names = {_norm(part.name) for part in path.parents}
    if 'train' in names:
        return 'train'
    if 'val' in names or 'validation' in names:
        return 'validation'
    if 'test' in names or 'eval' in names:
        return 'test'
    return None

def discover_records(dataset_root, dataset):
    root = Path(dataset_root)
    if not root.is_dir():
        print(f'[{dataset}] source directory not found: {root}')
        return []
    print(f'[{dataset}] scanning videos under {root}', flush=True)
    records, excluded = [], []
    video_paths = _video_files(root)
    print(f'[{dataset}] discovered {len(video_paths)} video files; hashing and mapping classes', flush=True)
    for index, path in enumerate(video_paths, start=1):
        label, source_class = _class_for(path, dataset)
        if label is None:
            excluded.append(path)
            continue
        relative = Path('datasets') / dataset / path.relative_to(root)
        records.append({
            'dataset': dataset, 'relative_path': relative.as_posix(), 'label': label,
            'source_class': source_class, 'source_group': path.stem,
            'official_split': _official_split(path), 'sha256': _sha256(path),
        })
        if index == len(video_paths) or index % 100 == 0:
            print(f'[{dataset}] processed {index}/{len(video_paths)} files; accepted={len(records)} excluded={len(excluded)}', flush=True)
    if excluded:
        known = {_norm(alias) for config in DATASET_REGISTRY.values() for alias in config['classes']} | {'weaponized', 'weaponizedviolence'}
        invalid = [path for path in excluded if not any(_norm(part.name) in known for part in (path, *path.parents))]
        if invalid:
            raise ValueError(f'{dataset}: files are not inside an approved class directory: {invalid[0]}')
        print(f'{dataset}: excluded {len(excluded)} files/classes outside the approved binary mapping')
    if records and any(record['official_split'] for record in records) and not all(record['official_split'] for record in records):
        raise ValueError(f'{dataset}: mixed official and fallback layouts; refusing to guess splits')
    return records

def copy_source_once(source, destination):
    source, destination = Path(source), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    paths = _video_files(source)
    print(f'copying {len(paths)} videos: {source} -> {destination}', flush=True)
    used_bytes = _dataset_bytes()
    budget_bytes = _effective_dataset_budget()
    print(f'dataset storage budget: {budget_bytes / 10**9:.2f} GB; already used: {used_bytes / 10**9:.2f} GB', flush=True)
    copied = skipped = 0
    for index, path in enumerate(paths, start=1):
        target = destination / path.relative_to(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            if _sha256(path) != _sha256(target):
                raise ValueError(f'existing destination differs from source: {target}; remove it explicitly before retrying')
            skipped += 1
        else:
            file_size = path.stat().st_size
            if used_bytes + file_size > budget_bytes:
                print(f'skipping {path}: dataset budget would be exceeded', flush=True)
                skipped += 1
                continue
            shutil.copy2(path, target)
            copied += 1
            used_bytes += file_size
        if index == len(paths) or index % 25 == 0:
            print(f'copy progress: {index}/{len(paths)}; imported={copied} reused={skipped}', flush=True)
    return copied, skipped

def _validated_existing_source(dataset):
    target = DATASETS_ROOT / dataset
    if not _video_files(target):
        return None
    try:
        records = discover_records(target, dataset)
    except ValueError as exc:
        print(f'{dataset}: existing Drive content rejected: {exc}')
        return None
    return target if records else None

def _kaggle_source(dataset, dataset_id):
    existing = _validated_existing_source(dataset)
    if existing is not None:
        print(f'{dataset}: reusing validated source {existing}', flush=True)
        return existing
    kaggle = shutil.which('kaggle')
    config_dir = Path(os.environ.get('KAGGLE_CONFIG_DIR', '~/.kaggle')).expanduser()
    has_credentials = bool(os.environ.get('KAGGLE_API_TOKEN')) or (config_dir / 'kaggle.json').is_file()
    if not kaggle or not has_credentials:
        print(f'{dataset}: skipped; configure Kaggle credentials and the kaggle CLI, or place an authorised source under {DATASETS_ROOT / dataset}')
        return None
    target = Path(tempfile.mkdtemp(prefix=f'{dataset}-'))
    print(f'{dataset}: downloading Kaggle dataset {dataset_id}', flush=True)
    subprocess.run([kaggle, 'datasets', 'download', '-d', dataset_id, '-p', str(target), '--unzip'], check=True)
    print(f'{dataset}: download complete at {target}', flush=True)
    return target

def _scvd_source():
    return _kaggle_source('scvd', SCVD_KAGGLE_DATASET)

def _ubi_fights_source():
    configured = os.environ.get('UBI_FIGHTS_SOURCE')
    if configured:
        return Path(configured).expanduser()
    return _kaggle_source('ubi_fights', UBI_FIGHTS_KAGGLE_DATASET)

def _surveillance_fight_source():
    configured = os.environ.get('SURVEILLANCE_FIGHT_SOURCE')
    if configured:
        return Path(configured).expanduser()
    existing = _validated_existing_source('surveillance_fight')
    if existing is not None:
        print(f'surveillance_fight: reusing validated source {existing}', flush=True)
        return existing
    target = Path(tempfile.mkdtemp(prefix='surveillance-fight-'))
    print(f'surveillance_fight: cloning {SURVEILLANCE_FIGHT_GITHUB_REPO}', flush=True)
    clone = subprocess.run(['git', 'clone', '--depth', '1', SURVEILLANCE_FIGHT_GITHUB_REPO, str(target)], capture_output=True, text=True)
    if clone.returncode != 0:
        print(f'surveillance_fight: download failed: {clone.stderr.strip()}', flush=True)
        raise RuntimeError('Surveillance Fight GitHub download failed')
    print(f'surveillance_fight: download complete at {target}', flush=True)
    return target

def _violent_flows_source():
    target = DATASETS_ROOT / 'violent_flows'
    existing = _validated_existing_source('violent_flows')
    if existing is not None:
        print(f'violent_flows: reusing validated source {existing}', flush=True)
        return existing
    configured = Path(os.environ.get('VIOLENT_FLOWS_SOURCE', str(VIOLENT_FLOWS_DEFAULT_SOURCE))).expanduser()
    if configured.is_dir():
        print(f'violent_flows: checking extracted source {configured}', flush=True)
        try:
            if discover_records(configured, 'violent_flows'):
                return configured
        except ValueError as exc:
            print(f'violent_flows: configured source rejected: {exc}')
    archive = Path(os.environ.get('VIOLENT_FLOWS_ARCHIVE', DRIVE_ROOT / 'movies.rar')).expanduser()
    unrar = shutil.which('unrar')
    if archive.is_file() and unrar:
        extracted = Path(tempfile.mkdtemp(prefix='violent-flows-'))
        print(f'violent_flows: extracting {archive}', flush=True)
        subprocess.run([unrar, 'x', '-o-', str(archive), str(extracted)], check=True)
        print(f'violent_flows: extraction complete at {extracted}', flush=True)
        return extracted
    print(f'violent_flows: skipped; place an authorised archive at {archive} or configure VIOLENT_FLOWS_ARCHIVE')
    return None

def prepare_sources():
    DATASETS_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'preparing dataset sources under {DATASETS_ROOT}', flush=True)
    sources = {'violent_flows': _violent_flows_source(), 'scvd': _scvd_source(), 'ubi_fights': _ubi_fights_source(), 'surveillance_fight': _surveillance_fight_source()}
    for dataset, source in sources.items():
        destination = DATASETS_ROOT / dataset
        if source is None:
            continue
        if source.resolve() != destination.resolve():
            try:
                copied, skipped = copy_source_once(source, destination)
            except Exception as exc:
                print(f'{dataset}: import failed: {exc}', flush=True)
                raise
            print(f'{dataset}: import complete; imported={copied} reused={skipped} destination={destination}', flush=True)
        else:
            print(f'{dataset}: using existing Drive files at {destination}', flush=True)
    return {dataset: DATASETS_ROOT / dataset for dataset in DATASET_REGISTRY}

## Manifest contract, deterministic splits, and checks

In [ ]:
def _split_groups(records):
    by_dataset = defaultdict(list)
    for record in records:
        by_dataset[record['dataset']].append(record)
    output = []
    for dataset, items in by_dataset.items():
        if dataset == 'violent_flows':
            output.extend({**item, 'split': 'external_test'} for item in items)
            continue
        official = {item.get('official_split') for item in items}
        if official and official <= {'train', 'validation', 'test'} and None not in official:
            output.extend({**item, 'split': item['official_split']} for item in items)
            continue
        groups = defaultdict(list)
        for item in items:
            groups[(item['label'], item['source_group'])].append(item)
        for label in ('normal', 'violent'):
            keys = sorted(key for key in groups if key[0] == label)
            random.Random(f'{SPLIT_SEED}:{dataset}:{label}').shuffle(keys)
            cut_train, cut_val = round(len(keys) * 0.7), round(len(keys) * 0.8)
            assignments = dict.fromkeys(keys[:cut_train], 'train')
            assignments.update(dict.fromkeys(keys[cut_train:cut_val], 'validation'))
            assignments.update(dict.fromkeys(keys[cut_val:], 'test'))
            output.extend({**item, 'split': assignments[key]} for key in keys for item in groups[key])
    return [{key: value for key, value in item.items() if key != 'official_split'} for item in output]

def deduplicate_records(records):
    seen_paths, seen_hashes, unique = set(), set(), []
    duplicates = 0
    for record in sorted(records, key=lambda row: (row['dataset'], row['relative_path'])):
        digest = record.get('sha256')
        if record['relative_path'] in seen_paths or (digest and digest in seen_hashes):
            duplicates += 1
            continue
        seen_paths.add(record['relative_path'])
        if digest:
            seen_hashes.add(digest)
        unique.append(record)
    if duplicates:
        print(f'removed {duplicates} duplicate-content/path records; retained {len(unique)} unique records', flush=True)
    return unique

def validate_records(records):
    seen_paths, seen_hashes, groups = set(), set(), {}
    for record in records:
        required = {'dataset', 'relative_path', 'label', 'split'}
        if not required <= record.keys():
            raise ValueError(f'missing manifest fields: {record}')
        relative = record['relative_path']
        path = Path(relative)
        if path.is_absolute() or '..' in path.parts or not relative.startswith('datasets/'):
            raise ValueError(f'unsafe Drive-relative path: {relative}')
        if record['dataset'] not in DATASET_REGISTRY or record['label'] not in {'normal', 'violent'}:
            raise ValueError(f'unsupported dataset/label: {record}')
        if record['split'] not in {'train', 'validation', 'test', 'external_test'}:
            raise ValueError(f'unsupported split: {record["split"]}')
        if record['dataset'] == 'violent_flows' and record['split'] != 'external_test':
            raise ValueError('violent_flows is external_test-only')
        digest = record.get('sha256')
        if record['relative_path'] in seen_paths or (digest and digest in seen_hashes):
            raise ValueError('duplicate path or content hash across manifest records')
        seen_paths.add(record['relative_path']); seen_hashes.add(digest) if digest else None
        group = (record.get('dataset'), record.get('source_group'))
        if group[1] and group in groups and groups[group] != record['split']:
            raise ValueError(f'source group leaked across splits: {group}')
        groups[group] = record['split']
    return records

def write_manifests(records, output_root=MANIFESTS_ROOT):
    validate_records(records)
    output_root = Path(output_root); output_root.mkdir(parents=True, exist_ok=True)
    names = {'train': 'train.json', 'validation': 'val.json', 'test': 'test.json', 'external_test': 'external_test.json'}
    for split, filename in names.items():
        rows = sorted((record for record in records if record['split'] == split), key=lambda row: (row['dataset'], row['relative_path']))
        payload = {'schema_version': '1.0', 'manifest_type': 'binary_video', 'split': split, 'seed': SPLIT_SEED, 'records': rows}
        (output_root / filename).write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    return output_root

def print_summary(records):
    counts = Counter((record['dataset'], record['split']) for record in records)
    print('Dataset                 Train   Val   Test   External')
    print('-' * 58)
    for dataset in DATASET_REGISTRY:
        train = counts[dataset, 'train']; validation = counts[dataset, 'validation']; test = counts[dataset, 'test']; external = counts[dataset, 'external_test']
        print(f'{dataset:<24} {train:>5} {validation:>5} {test:>6} {external:>9}')
    print('Total records:', len(records))

def contract_self_check():
    base = [{'dataset': 'ubi_fights', 'relative_path': f'datasets/ubi_fights/{label}-{index}.mp4', 'label': label, 'split': 'train', 'source_group': f'{label}-{index}', 'sha256': f'{index:064x}'} for index, label in enumerate(('normal', 'violent', 'normal', 'violent'))]
    first = _split_groups(base); second = _split_groups(base)
    assert first == second and all(row['split'] != 'external_test' for row in first)
    with tempfile.TemporaryDirectory() as directory:
        invalid_layout = Path(directory) / 'clip.mp4'; invalid_layout.write_bytes(b'fixture')
        try:
            discover_records(directory, 'scvd')
        except ValueError: pass
        else: raise AssertionError('invalid class layout accepted')
    try:
        validate_records([{**base[0], 'relative_path': '../escape.mp4'}])
    except ValueError: pass
    else: raise AssertionError('unsafe path accepted')
    try:
        validate_records([{**base[0], 'split': 'test'}, {**base[1], 'split': 'train', 'sha256': base[0]['sha256']}])
    except ValueError: pass
    else: raise AssertionError('duplicate content accepted')
    try:
        validate_records([{**base[0], 'source_group': 'same-group', 'split': 'train'}, {**base[1], 'source_group': 'same-group', 'split': 'test'}])
    except ValueError: pass
    else: raise AssertionError('source-group split leakage accepted')
    try:
        validate_records([{**base[0], 'dataset': 'violent_flows', 'split': 'train'}])
    except ValueError: pass
    else: raise AssertionError('violent_flows training row accepted')
    try:
        validate_records([{**base[0], 'label': 'weaponized'}])
    except ValueError: pass
    else: raise AssertionError('unsupported label accepted')
    duplicate_fixture = deduplicate_records([base[0], {**base[0], 'relative_path': 'datasets/ubi_fights/copy.mp4'}])
    assert len(duplicate_fixture) == 1
    with tempfile.TemporaryDirectory() as directory:
        source = Path(directory) / 'source'; destination = Path(directory) / 'destination'
        (source / 'normal').mkdir(parents=True); (destination / 'normal').mkdir(parents=True)
        (source / 'normal' / 'clip.mp4').write_bytes(b'source')
        (destination / 'normal' / 'clip.mp4').write_bytes(b'stale')
        try:
            copy_source_once(source, destination)
        except ValueError: pass
        else: raise AssertionError('stale destination accepted')
    print('media-independent manifest self-check: ok')

contract_self_check()

## Build manifests and inspect Drive usage

In [ ]:
def build_manifests():
    print('starting dataset preparation workflow', flush=True)
    roots = prepare_sources()
    discovered = []
    for dataset, root in roots.items():
        print(f'[{dataset}] building manifest records', flush=True)
        try:
            dataset_records = discover_records(root, dataset)
        except Exception as exc:
            print(f'[{dataset}] discovery failed: {exc}', flush=True)
            raise
        discovered.extend(dataset_records)
        print(f'[{dataset}] accepted records: {len(dataset_records)}', flush=True)
    print(f'total discovered records before splitting: {len(discovered)}', flush=True)
    discovered = deduplicate_records(discovered)
    records = _split_groups(discovered)
    used_bytes = _dataset_bytes()
    if used_bytes > MAX_DATASET_BYTES:
        raise RuntimeError(f'dataset storage exceeds {MAX_DATASET_GB:.2f} GB: {used_bytes / 10**9:.2f} GB')
    print(f'dataset storage after import: {used_bytes / 10**9:.2f} GB / {MAX_DATASET_GB:.2f} GB', flush=True)
    print(f'writing {len(records)} records to {MANIFESTS_ROOT}', flush=True)
    write_manifests(records)
    print_summary(records)
    usage = shutil.disk_usage(DRIVE_ROOT) if DRIVE_ROOT.exists() else None
    if usage:
        print({'dataset_budget_gb': round(MAX_DATASET_BYTES / 1e9, 2), 'drive_free_reserve_gb': DRIVE_FREE_RESERVE_GB, 'dataset_used_gb': round(_dataset_bytes() / 1e9, 2)}, flush=True)
        print({'drive_total_gb': round(usage.total / 1e9, 2), 'drive_used_gb': round(usage.used / 1e9, 2), 'drive_free_gb': round(usage.free / 1e9, 2)})
    print('manifests:', MANIFESTS_ROOT)
    return records

if RUN_WORKFLOW:
    all_records = build_manifests()
else:
    print('workflow skipped; set RUN_WORKFLOW = True for authorised Drive import and manifest generation')